In [7]:
%load_ext juliacall 

The juliacall extension is already loaded. To reload it, use:
  %reload_ext juliacall


In [9]:
""" Final code of Phase-1 with integrated Julia"""

from juliacall import Main as jl

jl.seval("""

function add(u::Vector{<:Number},v::Vector{<:Number})
        ulen = size(u)[1]
        vlen = size(v)[1]
        if ulen != vlen 
            throw(ArgumentError("dimension mismatch"))
        else 
            sum = Vector{promote_type(eltype(u),eltype(v))}(undef, ulen)
           @inbounds @simd for i = 1:1:ulen
                sum[i] = v[i]+u[i]
                           end
        return sum
        end
end
"""
 )

jl.seval( """

function sub(u::Vector{<:Number},v::Vector{<:Number})
        ulen = size(u)[1]
        vlen = size(v)[1]
        if ulen != vlen 
            throw(ArgumentError("dimension mismatch"))
        else 
            sum = Vector{promote_type(eltype(u),eltype(v))}(undef, ulen)
           @inbounds @simd for i = 1:1:ulen
                sum[i] = u[i]-v[i]
                           end
        return sum
        end
end
"""
 )

jl.seval("""

function scale(u::Vector{<:Number},sf::Number)
    scaled_vector = Vector{promote_type(eltype(u),eltype(sf))}(undef,size(u)[1])
    @inbounds @simd for i in 1:1:size(u)[1]
        scaled_vector[i] = sf*u[i]
    end
    return scaled_vector
end
""")
jl.seval("""
function mag(u::Vector{<:Number})
    a = 0
    @inbounds @simd for i in 1:1:size(u)[1]
        a += u[i]^2
    end
    return a^0.5
end
""")

jl.seval("""

function dot(u::Vector{<:Number},v::Vector{<:Number})
    ulen = size(u)[1]
    vlen = size(v)[1]
    if ulen != vlen
        throw(ArgumentError("Dimension Mismatch"))
    else 
        a = 0
        for i in 1:1:ulen
            a += u[i]*v[i]
        end
    return a
    end
end
""")



class vec():
    def __init__(self,*comp) : # comp stands for components
        self.__comp = jl.convert(jl.Vector[jl.Number],list(comp))
    def len(self): 
        return jl.size(self.__comp)[0]
    def get_vec(self):
        return self.__comp
    def __repr__(self):
        # Leveraging Julia's own function 
        return jl.sprint(jl.show, jl.MIME("text/plain"), self.__comp)
    def __add__(p1,p2) :
        return vec(*jl.add(p1.get_vec(),p2.get_vec()))
    def __sub__(p1,p2) :  # Gives p1-p2
        return vec(*jl.sub(p1.get_vec(),p2.get_vec())) 
    def scale(v,a = 1) :
        return vec(*jl.scale(v.get_vec(),a))
    def mag(v) :
       return jl.mag(v.get_vec())
    
    def dot(u,v) :
        return jl.dot(u.get_vec(),v.get_vec())
    def cosine(u,v):
        w=  vec.dot(u,v)/(u.mag()*v.mag())
        return   w
    def normalise(self) :
        w = self.mag()
        z = self.scale(1/w)
        return z
        
    


v1 = vec(1,2,3)
v2 = vec(4,5,6)

print(v1+v2)
print("-----------------------")
print(v1-v2)
print("-----------------------")
print(v1.mag())
print("-----------------------")
print(v1.dot(v2))
print("-----------------------")
print(v1.scale(2))
print("-----------------------")
print(v1.cosine(v2))
print("-----------------------")
print(v1.normalise())
print("-----------------------")
vec(1,2,3).dot(vec(1,2))

3-element Vector{Number}:
 5
 7
 9
-----------------------
3-element Vector{Number}:
 -3
 -3
 -3
-----------------------
3.7416573867739413
-----------------------
32
-----------------------
3-element Vector{Number}:
 2
 4
 6
-----------------------
0.9746318461970762
-----------------------
3-element Vector{Number}:
 0.2672612419124244
 0.5345224838248488
 0.8017837257372732
-----------------------


JuliaError: ArgumentError: Dimension Mismatch
Stacktrace:
 [1] dot(u::Vector{Number}, v::Vector{Number})
   @ Main .\none:5
 [2] pyjlany_call(self::typeof(dot), args_::Py, kwargs_::Py)
   @ PythonCall.JlWrap C:\Users\atharv shrivastav\.julia\packages\PythonCall\JksWe\src\JlWrap\any.jl:48
 [3] _pyjl_callmethod(f::Any, self_::Ptr{PythonCall.C.PyObject}, args_::Ptr{PythonCall.C.PyObject}, nargs::Int64)
   @ PythonCall.JlWrap C:\Users\atharv shrivastav\.julia\packages\PythonCall\JksWe\src\JlWrap\base.jl:71
 [4] _pyjl_callmethod(o::Ptr{PythonCall.C.PyObject}, args::Ptr{PythonCall.C.PyObject})
   @ PythonCall.JlWrap.Cjl C:\Users\atharv shrivastav\.julia\packages\PythonCall\JksWe\src\JlWrap\C.jl:72